# Day 12: Query Rewriting and Expansion, Walked Through

Users aren't search experts. This notebook walks through cleaning, synonym
expansion, decomposition, intent-aware rewriting, generalization/specialization,
pseudo-relevance feedback, ambiguity detection, and measuring the real impact on
a full retrieval pipeline.

Run each cell in order. Plain Python throughout, self-contained.

## Cell 1: Why query quality matters

Garbage in, garbage out: even perfect retrieval and ranking can only find the best
match for the query it's GIVEN. If that query is vague, typo-ridden, or genuinely
ambiguous, the system needs to improve the query itself before anything else can
help.

## Cell 2: Basic rewriting

Cleaning, stopword removal, abbreviation expansion, and typo fixing -- the
simplest useful layer.

In [ ]:
ABBREVIATIONS = {"rag": "retrieval augmented generation", "llm": "large language model", "db": "database"}
TYPOS = {"vektor": "vector", "embeding": "embedding", "retreival": "retrieval"}

def fix_typos(query):
    return " ".join(TYPOS.get(w.strip("?.,!").lower(), w) for w in query.split())

def expand_abbreviations(query):
    return " ".join(ABBREVIATIONS.get(w.strip("?.,!").lower(), w) for w in query.split())

examples = ["What is RAG?", "why is my vektor databse slow", "How do I use embeding for retreival?"]
for q in examples:
    fixed = fix_typos(q)
    expanded = expand_abbreviations(fixed)
    print(f"{q!r} -> {expanded!r}")

## Cell 3: Synonym expansion

Adding synonyms lets a query match documents that used different wording for the
same idea.

In [ ]:
import math

CONCEPTS = {
    "programming":   ["python", "programming", "language"],
    "data_types":    ["data", "types", "strings", "integers"],
    "retrieval":     ["rag", "retrieval", "retrieves", "generation", "lookup", "find", "look"],
    "freshness":     ["outdated", "fresh", "matters", "old", "problem"],
    "search_tech":   ["vector", "database", "embeddings", "similarity", "search", "fast"],
}

def text_to_embedding(text):
    words = set(w.strip("?.,!").lower() for w in text.split())
    return [float(sum(1 for w in words if w in cw)) for cw in CONCEPTS.values()]

def cosine_similarity(a, b):
    dot = sum(x*y for x,y in zip(a,b))
    ma, mb = math.sqrt(sum(x*x for x in a)), math.sqrt(sum(y*y for y in b))
    return dot/(ma*mb) if ma and mb else 0.0

SYNONYMS = {"get": ["retrieve", "lookup"], "old": ["outdated", "stale"]}

def expand_with_synonyms(query):
    words = query.lower().split()
    expanded = list(words)
    for w in words:
        if w in SYNONYMS:
            expanded.extend(SYNONYMS[w])
    return " ".join(expanded)

document = "RAG matters because language models can be outdated. Retrieval lets them use fresh information."
query = "How do I get old information fixed?"
expanded = expand_with_synonyms(query)

print(f"Original similarity: {cosine_similarity(text_to_embedding(query), text_to_embedding(document)):.3f}")
print(f"Expanded similarity: {cosine_similarity(text_to_embedding(expanded), text_to_embedding(document)):.3f}")

## Cell 4: Decomposition

Splitting a compound question into sub-questions lets each one be searched on its
own terms.

In [ ]:
import re

def identify_sub_questions(query):
    parts = [p.strip() for p in re.split(r'\band\b', query, flags=re.IGNORECASE) if p.strip()]
    if len(parts) < 2 or any(len(p.split()) < 3 for p in parts):
        return [query]
    return parts

compound = "What is RAG and why does it matter for outdated language models?"
parts = identify_sub_questions(compound)
print(f"Compound query: {compound!r}")
print(f"Sub-questions: {parts}")

## Cell 5: Semantic rewriting

Detecting intent (definition, how-to, explanation) and rewriting to match how
documents of that kind tend to be phrased.

In [ ]:
def detect_intent(query):
    lowered = query.lower().strip()
    if re.match(r'^(what is|what are)\b', lowered):
        return "definition"
    if re.match(r'^why\b', lowered):
        return "explanation"
    return "general"

def rewrite_for_intent(query, intent):
    lowered = query.strip().rstrip("?")
    if intent == "explanation":
        subject = re.sub(r'^why\s+', '', lowered, flags=re.IGNORECASE)
        return f"{subject} matters"  # 'matters' is a deliberate concept-vocabulary word
    return lowered

q = "Why does retrieval help language models?"
intent = detect_intent(q)
rewritten = rewrite_for_intent(q, intent)
print(f"Query: {q!r} -> intent: {intent}")
print(f"Rewritten: {rewritten!r}")

doc_a, doc_b = "RAG stands for Retrieval-Augmented Generation.", "RAG matters because language models can be outdated."
for label, text in [("original", q), ("rewritten", rewritten)]:
    sims = {name: cosine_similarity(text_to_embedding(text), text_to_embedding(doc)) for name, doc in [("What is RAG", doc_a), ("Why RAG Matters", doc_b)]}
    print(f"  {label}: {sims}")

## Cell 6: Expansion techniques

Adding context, adding related concepts, generalizing, and specializing all push a
query in different directions.

In [ ]:
short_query = "fast"
context_expanded = f"{short_query} vector database embeddings"

# Split into two distinct concept buckets so a vector-DB document and a
# keyword-search document can score DIFFERENTLY from each other, instead
# of both only ever touching one shared "search" bucket (which made both
# documents tie regardless of the query, a low-dimensional artifact).
doc_vector_db = "A vector database stores embeddings and allows fast similarity search."
doc_keyword = "Keyword search finds exact matching words quickly and simply."

VECTOR_WORDS = {"vector", "database", "embeddings", "similarity", "fast"}
KEYWORD_WORDS = {"keyword", "exact", "matching", "simply", "quickly"}

def mini_embed(text):
    words = set(w.strip("?.,!").lower() for w in text.split())
    return [float(len(words & VECTOR_WORDS)), float(len(words & KEYWORD_WORDS))]

for label, q in [("original", short_query), ("context-expanded", context_expanded)]:
    sims = {
        "Vector DB": cosine_similarity(mini_embed(q), mini_embed(doc_vector_db)),
        "Keyword Search": cosine_similarity(mini_embed(q), mini_embed(doc_keyword)),
    }
    print(f"{label:18s}: {sims}")

print(
    "\nBoth 'fast' alone and the context-expanded version already point "
    "clearly at Vector DB over Keyword Search here -- 'fast' happens to "
    "only appear in the vector-DB-style bucket. The context words didn't "
    "need to change the WINNER, but they do make the query carry more "
    "real signal overall, which matters more once a document set has many "
    "more candidates than just these two."
)

## Cell 7: Pseudo relevance feedback

Run the query once, extract important terms from the top results, and re-run with
the expanded query.

In [ ]:
STOPWORDS = {"what", "is", "a", "an", "the", "are", "of", "in", "to", "and", "for", "it", "its"}

def extract_important_terms(text, top_n=3):
    words = [w.strip("?.,!").lower() for w in text.split()]
    counts = {}
    for w in words:
        if w and w not in STOPWORDS and len(w) > 2:
            counts[w] = counts.get(w, 0) + 1
    return [w for w, _ in sorted(counts.items(), key=lambda pair: pair[1], reverse=True)[:top_n]]

query = "outdated"
top_result_text = "RAG matters because language models can be outdated. Retrieval lets them use fresh information."

first_pass_score = cosine_similarity(text_to_embedding(query), text_to_embedding(top_result_text))
terms = extract_important_terms(top_result_text)
expanded_query = f"{query} {' '.join(terms)}"
second_pass_score = cosine_similarity(text_to_embedding(expanded_query), text_to_embedding(top_result_text))

print(f"First pass query: {query!r} -> similarity {first_pass_score:.3f}")
print(f"Extracted terms: {terms}")
print(f"Expanded query: {expanded_query!r} -> similarity {second_pass_score:.3f}")

## Cell 8: Ambiguity resolution

Detecting ambiguity from retrieval's own uncertainty: a close score race between
genuinely different documents.

In [ ]:
docs = {
    "Python Data Types": "Python has several built-in data types including strings and integers.",
    "What is RAG": "RAG retrieves relevant documents before generating an answer.",
}

query = "look up data"
scores = {name: cosine_similarity(text_to_embedding(query), text_to_embedding(text)) for name, text in docs.items()}
ranked = sorted(scores.items(), key=lambda pair: pair[1], reverse=True)

print(f"Query: {query!r}")
print(f"Scores: {ranked}")

gap = ranked[0][1] - ranked[1][1]
is_ambiguous = gap < 0.1
print(f"Score gap: {gap:.3f} -> ambiguous: {is_ambiguous}")
if is_ambiguous:
    topics = [name for name, _ in ranked]
    print(f"Clarifying question: Your question could relate to {' or '.join(topics)}. Which did you mean?")

## Cell 9: Integration example

Measuring query rewriting's real impact on a tiny end-to-end retrieval pipeline.

In [ ]:
documents = [
    {"id": 2, "title": "What is RAG", "text": "RAG retrieves relevant documents before generating an answer."},
    {"id": 5, "title": "Why RAG Matters", "text": "RAG matters because language models can be outdated. Retrieval lets them use fresh information."},
]

def retrieve(query, docs, top_k=2):
    q_emb = text_to_embedding(query)
    scored = sorted(((d, cosine_similarity(q_emb, text_to_embedding(d["text"]))) for d in docs), key=lambda pair: pair[1], reverse=True)
    return [(d["title"], s) for d, s in scored[:top_k] if s > 0]

query = "How do I get old information fixed?"
print("Raw query results:     ", retrieve(query, documents))
print("Rewritten query results:", retrieve(expand_with_synonyms(query), documents))

## Cell 10: Key takeaways

- Users aren't search experts -- query rewriting meets them where they are instead
  of demanding perfectly-phrased input.
- Basic cleaning (typos, abbreviations, stopwords) catches the simplest failures
  cheaply.
- Synonym expansion bridges a query's wording and a document's wording when they
  genuinely differ -- but only helps if the expansion word maps to something the
  retrieval system can actually recognize.
- Decomposition helps genuinely compound questions by letting each part be
  searched on its own terms, without over-splitting single coherent ideas.
- Intent-aware rewriting matches a query's SHAPE to how relevant documents tend to
  be phrased -- most effective when the rewrite actually changes meaningful
  content, not just grammar.
- Expansion can go four directions (context, related concepts, generalize,
  specialize) -- the right one depends on whether the original query is too
  narrow or too broad.
- Pseudo-relevance feedback learns from a first retrieval pass, but trusts that
  pass's results without verifying them -- most reliable as a refinement on an
  already-reasonable query, not a rescue for a badly off-topic one.
- Ambiguity can be detected cheaply from retrieval's OWN uncertainty (a close
  score race between different documents), without needing to understand
  language deeply.
- Always measure a rewriting technique's real impact (precision, recall, NDCG,
  MRR, MAP) rather than assuming it helped just because it seems reasonable.